# Strategy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BEST_SMA = 60
BEST_ROC = 80
BEST_SL = 0.25
BEST_MA_S = 50
BEST_OFFSET = 2

def run_strategy():
    df = pd.read_excel('0205/data.xlsx', header=[0, 1], index_col=0)
    df.columns = df.columns.get_level_values(0)
    df = df.ffill().bfill()
    prices_arr = df.values
    dates = df.index
    sma_arr = df.rolling(BEST_SMA).mean().values
    roc_arr = df.pct_change(BEST_ROC).values
    ma_stop_arr = df.rolling(BEST_MA_S).mean().values
    
    initial_capital = 30000000.0
    slot_cash = [initial_capital / 3.0] * 3
    slot_holdings = [None] * 3 
    equity = []
    pending_exits = [None] * 3
    pending_entries = [None] * 3
    
    for i in range(len(dates)):
        curr_p = prices_arr[i]
        for s in range(3):
            if slot_holdings[s] is not None: slot_holdings[s]['max_p'] = max(slot_holdings[s]['max_p'], curr_p[slot_holdings[s]['idx']])
            if pending_exits[s] is not None:
                if slot_holdings[s] is not None: slot_cash[s] += slot_holdings[s]['shares'] * curr_p[slot_holdings[s]['idx']]; slot_holdings[s] = None
                pending_exits[s] = None
            if pending_entries[s] is not None:
                idx = pending_entries[s]; entry_p = curr_p[idx]; shares = slot_cash[s] // entry_p
                if shares > 0: slot_holdings[s] = {'idx': idx, 'shares': shares, 'max_p': entry_p}; slot_cash[s] -= shares * entry_p
                pending_entries[s] = None
        
        val = sum(slot_cash)
        for s in range(3):
            if slot_holdings[s] is not None: val += slot_holdings[s]['shares'] * curr_p[slot_holdings[s]['idx']]
        equity.append(val)
        
        for s in range(3):
            if slot_holdings[s] is not None:
                idx = slot_holdings[s]['idx']
                if curr_p[idx] < slot_holdings[s]['max_p'] * (1 - BEST_SL) or curr_p[idx] < ma_stop_arr[i, idx]: pending_exits[s] = "SL"
        if (i - BEST_OFFSET) % 5 == 0:
            eligible = (curr_p > sma_arr[i]) & (roc_arr[i] > 0)
            top_3 = np.where(eligible)[0][np.argsort(roc_arr[i, np.where(eligible)[0]])[::-1][:3]] if np.any(eligible) else []
            held = [h['idx'] if h is not None else -1 for h in slot_holdings]
            keep = [False]*3
            for s in range(3):
                if held[s] != -1 and held[s] in top_3 and pending_exits[s] is None: keep[s] = True
            for s in range(3):
                if not keep[s] and slot_holdings[s] is not None and pending_exits[s] is None: pending_exits[s] = "Exit"
            new_t = [t for t in top_3 if t not in [held[s] for s in range(3) if keep[s]]]
            ti = 0
            for s in range(3):
                if not keep[s] and pending_entries[s] is None and ti < len(new_t): pending_entries[s] = new_t[ti]; ti += 1
    return pd.Series(equity, index=dates)

equity_curve = run_strategy()
equity_curve.plot(title='Equity Curve')
plt.show()
print(f"Final Calmar: {(equity_curve.iloc[-1]/30000000)**(252/len(equity_curve))-1 / abs((equity_curve/equity_curve.cummax()-1).min()):.2f}")
